In [61]:
import pandas as pd
import numpy as np 

## Import Datasets 

In [62]:
# 1. Load the CSV files
df_postcodes = pd.read_csv(r"D:\housing_csv\clean_postcodes.csv")
df_registry  = pd.read_csv(r"D:\housing_csv\Market T - PriceRegistory.csv")
df_pop_density = pd.read_csv(r"D:\housing_csv\Population Density.csv")
df_crime = pd.read_csv(r"D:\housing_csv\Crime rate.csv")
df_osm = pd.read_csv(r"D:\housing_csv\london_osm.csv")

# Quick check: print the names of all loaded datasets
datasets = {
    "Postcodes": df_postcodes,
    "Registry": df_registry,
    "Pop Density": df_pop_density,
    "Crime": df_crime,
    "OSM": df_osm
,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

Postcodes: 332307 rows, 5 columns
Registry: 12015 rows, 5 columns
Pop Density: 24960 rows, 9 columns
Crime: 18688 rows, 30 columns
OSM: 1830 rows, 5 columns


In [63]:
df_postcodes.head()

,postcode,latitude,longitude,district,county
0,BR1 1AA,51.401545,0.015440,Bromley,Greater London
1,BR1 1AB,51.406333,0.015234,Bromley,Greater London
2,BR1 1AD,51.400057,0.016741,Bromley,Greater London
3,BR1 1AE,51.404543,0.014221,Bromley,Greater London
4,BR1 1AF,51.401391,0.014973,Bromley,Greater London


In [64]:
df_registry.head()

,Code,Area,Year,Measure,Value
0,E09000001,City of London,Year ending Dec 1995,Median,"105,000"
1,E09000002,Barking and Dagenham,Year ending Dec 1995,Median,"49,000"
2,E09000003,Barnet,Year ending Dec 1995,Median,"85,125"
3,E09000004,Bexley,Year ending Dec 1995,Median,"62,000"
4,E09000005,Brent,Year ending Dec 1995,Median,"68,000"


In [65]:
df_pop_density.head()

,Code,Borough,Ward_Name,Year,Population,Hectares,Square_Kilometres,Population_per_hectare,Population_per_square_kilometre
0,E05000026,Barking and Dagenham,Abbey,2011,12904,127.9,1.279,100.891321,10089.132130
1,E05000027,Barking and Dagenham,Alibon,2011,10468,136.1,1.361,76.914034,7691.403380
2,E05000028,Barking and Dagenham,Becontree,2011,11638,128.4,1.284,90.638629,9063.862928
3,E05000029,Barking and Dagenham,Chadwell Heath,2011,10098,338.0,3.380,29.875740,2987.573964
4,E05000030,Barking and Dagenham,Eastbrook,2011,10581,345.4,3.454,30.634047,3063.404748


In [66]:
df_crime.head()

,MajorText,MinorText,WardName,WardCode,LookUp_BoroughName,202402,202403,202404,202405,202406,...,202505,202506,202507,202508,202509,202510,202511,202512,202601,202602
0,ARSON AND CRIMINAL DAMAGE,ARSON,Heathrow Villages,E05013570,Aviation Security (SO18),0,2,0,0,1,...,0,5,3,2,0,0,1,3,1,0
1,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE,Heathrow Villages,E05013570,Aviation Security (SO18),0,21,31,24,22,...,15,18,25,24,30,26,24,40,29,18
2,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,Heathrow Villages,E05013570,Aviation Security (SO18),0,6,3,3,0,...,6,4,1,4,2,5,2,3,0,9
3,BURGLARY,RES BURGLARY OF A HOME,Heathrow Villages,E05013570,Aviation Security (SO18),0,3,1,2,5,...,1,8,1,4,2,3,2,5,4,3
4,BURGLARY,RES BURGLARY OF UNCONNECTED BUILDING,Heathrow Villages,E05013570,Aviation Security (SO18),0,1,1,0,0,...,0,1,0,2,1,0,1,1,2,1


In [67]:
df_osm.head()

,lat,lon,category,type,name
0,44.316171,-72.113429,amenity,school,Four Corners School
1,44.277838,-72.157596,amenity,school,Walter Harvey School
2,39.957458,-82.942063,amenity,restaurant,Moshi Sushi
3,39.957365,-82.938288,amenity,restaurant,Giuseppe's Ritrovo
4,39.957313,-82.931961,amenity,restaurant,Noahla


### 1. Postcode Data Analysis

In [68]:
df_postcodes.dtypes

postcode      object
latitude     float64
longitude    float64
district      object
county        object
dtype: object

In [69]:
df_postcodes[df_postcodes.duplicated()]
df_postcodes.isnull().sum()

postcode     0
latitude     0
longitude    0
district     0
county       0
dtype: int64

In [70]:
df_postcodes.columns
df_postcodes['district'] = df_postcodes['district'].astype('string')
df_postcodes = df_postcodes.drop(columns=['county'])

### 2. Registory Analysis

In [71]:
df_registry.dtypes

Code       object
Area       object
Year       object
Measure    object
Value      object
dtype: object

In [72]:
df_registry[df_postcodes.duplicated()]
df_registry.isnull().sum()

C:\Users\saket\AppData\Local\Temp\ipykernel_5480\2467081338.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_registry[df_postcodes.duplicated()]


Code       0
Area       0
Year       0
Measure    0
Value      0
dtype: int64

In [73]:
#Rename columns for consistency
df_registry = df_registry.rename(columns={
    "Code": "code",
    "Area": "area",
    "Year": "year",
    "Measure": "measure",
    "Value": "value"
})

# Remove commas and convert value to numeric
df_registry["value"] = (
    df_registry["value"]
    .astype(str)
    .str.replace(",", "")
    .astype(float)
)

# Convert to category where useful
df_registry["code"] = df_registry["code"].astype("category")
df_registry["area"] = df_registry["area"].astype("category")
df_registry["measure"] = df_registry["measure"].astype("category")

# Extract numeric year (e.g., from "Year ending Dec 1995")
df_registry["year"] = df_registry["year"].str.extract(r"(\d{4})").astype(int)

In [74]:
df_registry.head()

,code,area,year,measure,value
0,E09000001,City of London,1995,Median,105000.0
1,E09000002,Barking and Dagenham,1995,Median,49000.0
2,E09000003,Barnet,1995,Median,85125.0
3,E09000004,Bexley,1995,Median,62000.0
4,E09000005,Brent,1995,Median,68000.0


### 3. Population Data Cleaning  

In [75]:
# 1. Lowercase column names
df_pop_density.columns = df_pop_density.columns.str.lower()

# 2. Strip whitespace (just in case)
df_pop_density.columns = df_pop_density.columns.str.strip()

# 3. Check current data types
print(df_pop_density.dtypes)

# 4. Convert data types explicitly
df_pop_density = df_pop_density.astype({
    'code': 'string',
    'borough': 'string',
    'ward_name': 'string',
    'year': 'int64',
    'population': 'int64',
    'hectares': 'float64',
    'square_kilometres': 'float64',
    'population_per_hectare': 'float64',
    'population_per_square_kilometre': 'float64'
})

# 5. Check for null values
print(df_pop_density.isnull().sum())

# Optional: handle nulls (choose one strategy)
# Drop rows with nulls
df_pop_density = df_pop_density.dropna()

# OR fill nulls (example)
# df_pop_density = df_pop_density.fillna(0)

# 6. Check for duplicates
print(df_pop_density.duplicated().sum())

# 7. Remove duplicates
df_pop_density = df_pop_density.drop_duplicates()

# 8. Reset index after cleaning
df_pop_density = df_pop_density.reset_index(drop=True)

# Final check
print(df_pop_density.info())

code                                object
borough                             object
ward_name                           object
year                                 int64
population                           int64
hectares                           float64
square_kilometres                  float64
population_per_hectare             float64
population_per_square_kilometre    float64
dtype: object
code                               0
borough                            0
ward_name                          0
year                               0
population                         0
hectares                           0
square_kilometres                  0
population_per_hectare             0
population_per_square_kilometre    0
dtype: int64
0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24960 entries, 0 to 24959
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   code          

In [76]:
df_pop_density.head()

,code,borough,ward_name,year,population,hectares,square_kilometres,population_per_hectare,population_per_square_kilometre
0,E05000026,Barking and Dagenham,Abbey,2011,12904,127.9,1.279,100.891321,10089.132130
1,E05000027,Barking and Dagenham,Alibon,2011,10468,136.1,1.361,76.914034,7691.403380
2,E05000028,Barking and Dagenham,Becontree,2011,11638,128.4,1.284,90.638629,9063.862928
3,E05000029,Barking and Dagenham,Chadwell Heath,2011,10098,338.0,3.380,29.875740,2987.573964
4,E05000030,Barking and Dagenham,Eastbrook,2011,10581,345.4,3.454,30.634047,3063.404748


### 4. Crime Data Cleaning 

In [77]:
# 1. Standardise column names (lowercase + strip spaces)
df_crime.columns = df_crime.columns.str.lower().str.strip()

# 2. Lowercase all string/object columns
for col in df_crime.select_dtypes(include='object').columns:
    df_crime[col] = df_crime[col].str.lower().str.strip()

# 3. Remove duplicate rows
df_crime = df_crime.drop_duplicates()

# 4. Handle nulls
# Option A: fill with sensible defaults
df_crime = df_crime.fillna({
    col: 0 if df_crime[col].dtype != 'object' else 'unknown'
    for col in df_crime.columns
})

# (Optional) If you prefer dropping rows with nulls instead:
# df_crime = df_crime.dropna()

# 5. Fix dtypes
# Convert numeric columns safely
for col in df_crime.columns:
    df_crime[col] = pd.to_numeric(df_crime[col], errors='ignore')

# Optional: explicitly convert date-like columns (YYYYMM format)
date_cols = [col for col in df_crime.columns if col.isdigit()]
df_crime[date_cols] = df_crime[date_cols].apply(pd.to_numeric, errors='coerce')

# 6. Final sanity check
print(df_crime.info())
print(df_crime.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18688 entries, 0 to 18687
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   majortext           18688 non-null  object
 1   minortext           18688 non-null  object
 2   wardname            18688 non-null  object
 3   wardcode            18688 non-null  object
 4   lookup_boroughname  18688 non-null  object
 5   202402              18688 non-null  int64 
 6   202403              18688 non-null  int64 
 7   202404              18688 non-null  int64 
 8   202405              18688 non-null  int64 
 9   202406              18688 non-null  int64 
 10  202407              18688 non-null  int64 
 11  202408              18688 non-null  int64 
 12  202409              18688 non-null  int64 
 13  202410              18688 non-null  int64 
 14  202411              18688 non-null  int64 
 15  202412              18688 non-null  int64 
 16  202501              18

In [78]:
print(df_crime.head())

                   majortext                             minortext  \
0  arson and criminal damage                                 arson   
1  arson and criminal damage                       criminal damage   
2                   burglary       burglary business and community   
3                   burglary                res burglary of a home   
4                   burglary  res burglary of unconnected building   

            wardname   wardcode        lookup_boroughname  202402  202403  \
0  heathrow villages  e05013570  aviation security (so18)       0       2   
1  heathrow villages  e05013570  aviation security (so18)       0      21   
2  heathrow villages  e05013570  aviation security (so18)       0       6   
3  heathrow villages  e05013570  aviation security (so18)       0       3   
4  heathrow villages  e05013570  aviation security (so18)       0       1   

   202404  202405  202406  ...  202505  202506  202507  202508  202509  \
0       0       0       1  ...       0    

In [79]:
# Identify date columns
date_cols = [col for col in df_crime.columns if col.isdigit()]

# Melt
df_long = df_crime.melt(
    id_vars=[col for col in df_crime.columns if col not in date_cols],
    value_vars=date_cols,
    var_name="year_month",
    value_name="crime_count"
)

# Convert to datetime (keep as single column)
df_long["year_month"] = pd.to_datetime(df_long["year_month"], format="%Y%m")
print(df_long.head())

                   majortext                             minortext  \
0  arson and criminal damage                                 arson   
1  arson and criminal damage                       criminal damage   
2                   burglary       burglary business and community   
3                   burglary                res burglary of a home   
4                   burglary  res burglary of unconnected building   

            wardname   wardcode        lookup_boroughname year_month  \
0  heathrow villages  e05013570  aviation security (so18) 2024-02-01   
1  heathrow villages  e05013570  aviation security (so18) 2024-02-01   
2  heathrow villages  e05013570  aviation security (so18) 2024-02-01   
3  heathrow villages  e05013570  aviation security (so18) 2024-02-01   
4  heathrow villages  e05013570  aviation security (so18) 2024-02-01   

   crime_count  
0            0  
1            0  
2            0  
3            0  
4            0  


In [80]:
df_long["crime_count"].describe()

count    467200.000000
mean          3.814585
std          14.177346
min           0.000000
25%           0.000000
50%           1.000000
75%           4.000000
max        2402.000000
Name: crime_count, dtype: float64

In [81]:
df_long.head()

,majortext,minortext,wardname,wardcode,lookup_boroughname,year_month,crime_count
0,arson and criminal damage,arson,heathrow villages,e05013570,aviation security (so18),2024-02-01,0
1,arson and criminal damage,criminal damage,heathrow villages,e05013570,aviation security (so18),2024-02-01,0
2,burglary,burglary business and community,heathrow villages,e05013570,aviation security (so18),2024-02-01,0
3,burglary,res burglary of a home,heathrow villages,e05013570,aviation security (so18),2024-02-01,0
4,burglary,res burglary of unconnected building,heathrow villages,e05013570,aviation security (so18),2024-02-01,0


### 5. Osm Data cleaning

In [82]:
import pandas as pd

# 1. Standardise column names
df_osm.columns = df_osm.columns.str.lower().str.strip()

# 2. Rename for clarity (optional but recommended)
df_osm = df_osm.rename(columns={
    "category": "osm_category",
    "type": "osm_type",
    "name": "place_name"
})

# 3. Remove duplicates
df_osm = df_osm.drop_duplicates()

# 4. Handle missing values
# Fill names with 'unknown', keep others as-is or drop if critical
df_osm["place_name"] = df_osm["place_name"].fillna("unknown")

# If you want to drop rows missing key info (lat/lon/type):
df_osm = df_osm.dropna(subset=["lat", "lon", "osm_type"])

# 5. Fix data types
df_osm["lat"] = pd.to_numeric(df_osm["lat"], errors="coerce")
df_osm["lon"] = pd.to_numeric(df_osm["lon"], errors="coerce")

# 6. Remove invalid coordinates
df_osm = df_osm[
    (df_osm["lat"].between(-90, 90)) &
    (df_osm["lon"].between(-180, 180))
]

# 7. Standardise text columns
for col in ["osm_category", "osm_type", "place_name"]:
    df_osm[col] = df_osm[col].str.lower().str.strip()

# 8. Final check
print(df_osm.info())
print(df_osm.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1830 entries, 0 to 1829
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   lat           1830 non-null   float64
 1   lon           1830 non-null   float64
 2   osm_category  1830 non-null   object 
 3   osm_type      1830 non-null   object 
 4   place_name    1830 non-null   object 
dtypes: float64(2), object(3)
memory usage: 71.6+ KB
None
         lat        lon osm_category    osm_type            place_name
0  44.316171 -72.113429      amenity      school   four corners school
1  44.277838 -72.157596      amenity      school  walter harvey school
2  39.957458 -82.942063      amenity  restaurant           moshi sushi
3  39.957365 -82.938288      amenity  restaurant    giuseppe's ritrovo
4  39.957313 -82.931961      amenity  restaurant                noahla


In [83]:
df_osm["osm_type"].unique()

array(['school', 'restaurant', 'fitness_centre', 'bus_stop',
       'supermarket', 'station', 'subway_entrance', 'prep_school',
       'music_school', 'driving_school', 'hospital', 'dancing_school',
       'bus_station', 'hospital_department', 'train_station_entrance'],
      dtype=object)

In [84]:
print(df_postcodes.columns)
print(df_registry.columns)
print(df_pop_density.columns)
print(df_long.columns)
print(df_osm.columns)

Index(['postcode', 'latitude', 'longitude', 'district'], dtype='object')
Index(['code', 'area', 'year', 'measure', 'value'], dtype='object')
Index(['code', 'borough', 'ward_name', 'year', 'population', 'hectares',
       'square_kilometres', 'population_per_hectare',
       'population_per_square_kilometre'],
      dtype='object')
Index(['majortext', 'minortext', 'wardname', 'wardcode', 'lookup_boroughname',
       'year_month', 'crime_count'],
      dtype='object')
Index(['lat', 'lon', 'osm_category', 'osm_type', 'place_name'], dtype='object')


In [85]:
df_osm.head()

,lat,lon,osm_category,osm_type,place_name
0,44.316171,-72.113429,amenity,school,four corners school
1,44.277838,-72.157596,amenity,school,walter harvey school
2,39.957458,-82.942063,amenity,restaurant,moshi sushi
3,39.957365,-82.938288,amenity,restaurant,giuseppe's ritrovo
4,39.957313,-82.931961,amenity,restaurant,noahla


In [86]:
df_postcodes.to_csv("df_postcodes.csv", index=False)

In [87]:
import os
print(os.listdir())

['.gitkeep', 'df_osm.csv', 'df_postcodes.csv', 'housing.ipynb']


In [88]:
df_osm.to_csv("df_osm.csv", index=False)    

In [89]:
df_registry.head()

,code,area,year,measure,value
0,E09000001,City of London,1995,Median,105000.0
1,E09000002,Barking and Dagenham,1995,Median,49000.0
2,E09000003,Barnet,1995,Median,85125.0
3,E09000004,Bexley,1995,Median,62000.0
4,E09000005,Brent,1995,Median,68000.0


### Feature Engineering 

In [90]:
df_postcodes["postcode"] = (
    df_postcodes["postcode"]
    .str.upper()
    .str.replace(" ", "", regex=False)
)

In [91]:
df_registry["area"].unique()

['City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', ..., 'South East', 'South West', 'England', 'Wales', 'England And Wales']
Length: 45
Categories (45, object): ['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', ..., 'Wandsworth', 'West Midlands', 'Westminster', 'Yorkshire And The Humber']

In [92]:
london_boroughs = [
    "CAMDEN", "GREENWICH", "HACKNEY", "HAMMERSMITH AND FULHAM",
    "ISLINGTON", "KENSINGTON AND CHELSEA", "LAMBETH", "LEWISHAM",
    "SOUTHWARK", "TOWER HAMLETS", "WANDSWORTH", "WESTMINSTER",
    "BARKING AND DAGENHAM", "BARNET", "BEXLEY", "BRENT",
    "BROMLEY", "CROYDON", "EALING", "ENFIELD", "HARINGEY",
    "HARROW", "HAVERING", "HILLINGDON", "HOUNSLOW",
    "KINGSTON UPON THAMES", "MERTON", "NEWHAM",
    "REDBRIDGE", "RICHMOND UPON THAMES",
    "SUTTON", "WALTHAM FOREST",
    "CITY OF LONDON"
]
df_registry["area"] = df_registry["area"].str.upper().str.strip()

df_registry = df_registry[df_registry["area"].isin(london_boroughs)]


In [93]:
df_registry["measure"].unique()

['Median', 'Mean', 'Sales']
Categories (3, object): ['Mean', 'Median', 'Sales']

In [120]:
df_price = df_registry[df_registry["measure"] == "Mean"]
df_price.head()

,code,area,year,measure,value
4005,E09000001,CITY OF LONDON,1995,Mean,146043.0
4006,E09000002,BARKING AND DAGENHAM,1995,Mean,50615.0
4007,E09000003,BARNET,1995,Mean,111019.0
4008,E09000004,BEXLEY,1995,Mean,66294.0
4009,E09000005,BRENT,1995,Mean,77258.0


In [121]:
df_price = df_price.groupby("area").agg({
    "value": "mean"
}).reset_index()

df_price = df_price.rename(columns={
    "area": "borough_name",
    "value": "avg_price"
})

In [97]:
df_population = df_pop_density.groupby("code").agg({
    "population": "sum",
    "hectares": "sum",
    "population_per_hectare": "mean"
}).reset_index()

df_population = df_population.rename(columns={
    "code": "borough_code"
})

In [98]:
df_main = df_borough.merge(df_price, on="borough_name", how="left")

In [99]:
df_population.shape

(624, 4)

In [100]:
# Add Price Data (Registry)
df_registry.columns = df_registry.columns.str.lower().str.strip()

df_price = df_registry.groupby("code").agg({
    "value": "mean"
}).reset_index()

df_price = df_price.rename(columns={
    "code": "borough_code",
    "value": "avg_price"
})

In [122]:
df_main = df_main.merge(df_price, on="borough_name", how="left")

In [123]:
df_main.head()

,borough_code,borough_name,avg_price_x,population,hectares,population_per_hectare,crime_count,amenity_count,amenity_diversity,crime_per_capita,avg_price_y
0,E05000026,BARKING AND DAGENHAM,NaN,791030,5116.0,154.618843,39156,NaN,NaN,0.049500,154721.94382
1,E05000027,BARKING AND DAGENHAM,NaN,466271,5444.0,85.648604,39156,NaN,NaN,0.083977,154721.94382
2,E05000028,BARKING AND DAGENHAM,NaN,566698,5136.0,110.338396,39156,NaN,NaN,0.069095,154721.94382
3,E05000029,BARKING AND DAGENHAM,NaN,449004,13520.0,33.210355,39156,NaN,NaN,0.087206,154721.94382
4,E05000030,BARKING AND DAGENHAM,NaN,453890,13816.0,32.852490,39156,NaN,NaN,0.086268,154721.94382


In [103]:
# Add Population Features
df_population = df_pop_density.groupby("code").agg({
    "population": "sum",
    "hectares": "sum",
    "population_per_hectare": "mean"
}).reset_index()

df_population = df_population.rename(columns={
    "code": "borough_code"
})

In [104]:
#Merge 
df_main = df_main.merge(df_population, on="borough_code", how="left")

In [105]:
# Add crime features
df_long.columns = df_long.columns.str.lower().str.strip()

df_long["lookup_boroughname"] = df_long["lookup_boroughname"].str.upper().str.strip()
df_main["borough_name"] = df_main["borough_name"].str.upper().str.strip()

In [106]:
#Aggregate crime data by borough
df_crime_agg = df_long.groupby("lookup_boroughname").agg({
    "crime_count": "sum"
}).reset_index()

df_crime_agg = df_crime_agg.rename(columns={
    "lookup_boroughname": "borough_name"
})

In [107]:
#Merge with borough table
df_main = df_main.merge(df_crime_agg, on="borough_name", how="left")

In [108]:
# add OSM features (e.g., count of amenities)
df_borough_coords = df_postcodes.groupby("district").agg({
    "latitude": "mean",
    "longitude": "mean"
}).reset_index()

df_borough_coords = df_borough_coords.rename(columns={
    "district": "borough_name"
})

In [109]:
# Get borough coordinates from postcodes (average lat/lon of all postcodes in each borough)
df_postcodes.columns = df_postcodes.columns.str.lower().str.strip()

df_postcodes["district"] = df_postcodes["district"].str.upper().str.strip()

df_borough_coords = df_postcodes.groupby("district").agg({
    "latitude": "mean",
    "longitude": "mean"
}).reset_index()

df_borough_coords = df_borough_coords.rename(columns={
    "district": "borough_name"
})

In [110]:
df_osm.columns = df_osm.columns.str.lower().str.strip()

df_osm = df_osm.rename(columns={
    "lat": "latitude",
    "lon": "longitude"
})

In [111]:
# Assign borough coordinates to Nearest Borough in OSM data


def find_nearest_borough(row, borough_df):
    distances = (
        (borough_df["latitude"] - row["latitude"])**2 +
        (borough_df["longitude"] - row["longitude"])**2
    )
    return borough_df.loc[distances.idxmin(), "borough_name"]

df_osm["borough_name"] = df_osm.apply(
    lambda row: find_nearest_borough(row, df_borough_coords),
    axis=1
)

In [112]:
#Aggregate Ameneities by Borough
df_amenities = df_osm.groupby("borough_name").agg({
    "place_name": "count",
    "osm_category": "nunique"
}).reset_index()

df_amenities = df_amenities.rename(columns={
    "place_name": "amenity_count",
    "osm_category": "amenity_diversity"
})

In [113]:
df_main = df_main.merge(df_amenities, on="borough_name", how="left")

In [117]:
df_main.head()

,borough_code,borough_name,avg_price,population,hectares,population_per_hectare,crime_count,amenity_count,amenity_diversity,crime_per_capita
0,E05000026,BARKING AND DAGENHAM,NaN,791030,5116.0,154.618843,39156,NaN,NaN,0.049500
1,E05000027,BARKING AND DAGENHAM,NaN,466271,5444.0,85.648604,39156,NaN,NaN,0.083977
2,E05000028,BARKING AND DAGENHAM,NaN,566698,5136.0,110.338396,39156,NaN,NaN,0.069095
3,E05000029,BARKING AND DAGENHAM,NaN,449004,13520.0,33.210355,39156,NaN,NaN,0.087206
4,E05000030,BARKING AND DAGENHAM,NaN,453890,13816.0,32.852490,39156,NaN,NaN,0.086268


In [115]:
#Crime 
df_main["crime_per_capita"] = df_main["crime_count"] / df_main["population"]

In [ ]:
df_main.describe()

,avg_price,population,hectares,population_per_hectare,crime_count,amenity_count,amenity_diversity,crime_per_capita
count,0.0,6.240000e+02,624.000000,624.000000,624.000000,135.000000,135.000000,624.000000
mean,NaN,6.377803e+05,10202.352564,95.900460,55809.791667,218.481481,2.807407,0.091788
std,NaN,2.038644e+05,10329.764233,56.917828,25884.919016,330.214310,1.998120,0.049797
min,NaN,2.190210e+05,1564.000000,1.885836,22709.000000,1.000000,1.000000,0.020286
25%,NaN,5.115120e+05,4923.000000,52.823195,41214.000000,8.000000,1.000000,0.065355
50%,NaN,6.077875e+05,7430.000000,82.433671,54354.000000,15.000000,1.000000,0.082616
75%,NaN,7.050372e+05,11512.000000,133.412480,64055.000000,230.000000,5.000000,0.104040
max,NaN,2.202376e+06,116140.000000,399.548423,167568.000000,935.000000,5.000000,0.452197


In [ ]:
df_main.head()

,borough_code,borough_name,population,hectares,population_per_hectare,crime_count,amenity_count,amenity_diversity,crime_per_capita,avg_price
0,E05000026,BARKING AND DAGENHAM,791030,5116.0,154.618843,39156,NaN,NaN,0.049500,154721.94382
1,E05000027,BARKING AND DAGENHAM,466271,5444.0,85.648604,39156,NaN,NaN,0.083977,154721.94382
2,E05000028,BARKING AND DAGENHAM,566698,5136.0,110.338396,39156,NaN,NaN,0.069095,154721.94382
3,E05000029,BARKING AND DAGENHAM,449004,13520.0,33.210355,39156,NaN,NaN,0.087206,154721.94382
4,E05000030,BARKING AND DAGENHAM,453890,13816.0,32.852490,39156,NaN,NaN,0.086268,154721.94382


In [128]:
cols = df_main.columns.tolist()

# remove avg_price from current position
cols.remove("avg_price")

# insert it after borough_name
index = cols.index("borough_name") + 1
cols.insert(index, "avg_price")

df_main = df_main[cols]

In [129]:
df_main.head()

,borough_code,borough_name,avg_price,population,hectares,population_per_hectare,crime_count,amenity_count,amenity_diversity,crime_per_capita
0,E05000026,BARKING AND DAGENHAM,154721.94382,791030,5116.0,154.618843,39156,NaN,NaN,0.049500
1,E05000027,BARKING AND DAGENHAM,154721.94382,466271,5444.0,85.648604,39156,NaN,NaN,0.083977
2,E05000028,BARKING AND DAGENHAM,154721.94382,566698,5136.0,110.338396,39156,NaN,NaN,0.069095
3,E05000029,BARKING AND DAGENHAM,154721.94382,449004,13520.0,33.210355,39156,NaN,NaN,0.087206
4,E05000030,BARKING AND DAGENHAM,154721.94382,453890,13816.0,32.852490,39156,NaN,NaN,0.086268


In [130]:
df_main["borough_name"] = df_main["borough_name"].str.title()

In [131]:
df_main.head()

,borough_code,borough_name,avg_price,population,hectares,population_per_hectare,crime_count,amenity_count,amenity_diversity,crime_per_capita
0,E05000026,Barking And Dagenham,154721.94382,791030,5116.0,154.618843,39156,NaN,NaN,0.049500
1,E05000027,Barking And Dagenham,154721.94382,466271,5444.0,85.648604,39156,NaN,NaN,0.083977
2,E05000028,Barking And Dagenham,154721.94382,566698,5136.0,110.338396,39156,NaN,NaN,0.069095
3,E05000029,Barking And Dagenham,154721.94382,449004,13520.0,33.210355,39156,NaN,NaN,0.087206
4,E05000030,Barking And Dagenham,154721.94382,453890,13816.0,32.852490,39156,NaN,NaN,0.086268


In [132]:
df_main.to_csv("borough_data.csv", index=False)